In [1]:
import torch
import esm

# 强制使用 GPU
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 加载模型
print("加载 ESMFold 模型...")
model = esm.pretrained.esmfold_v1()
model = model.eval().to(device)  # 确保移动到 GPU
print("✓ 模型加载成功")

# 测试序列
sequence = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
print(f"预测序列: {sequence} (长度: {len(sequence)})")

# 预测结构
with torch.no_grad():
    output = model.infer_pdb(sequence)

# 保存结果
with open("result.pdb", "w") as f:
    f.write(output)
print("✓ PDB 文件已保存: result.pdb")

# 手动计算 pLDDT（不使用 biotite）
with torch.no_grad():
    output_data = model.infer(sequence)
    plddt = output_data["plddt"].mean().item()
    print(f"平均 pLDDT (置信度): {plddt:.2f}")

使用设备: cuda:1
GPU: NVIDIA GeForce RTX 4090
加载 ESMFold 模型...
✓ 模型加载成功
预测序列: MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG (长度: 65)
✓ PDB 文件已保存: result.pdb
平均 pLDDT (置信度): 80.41


In [1]:
from pathlib import Path
import subprocess
import tempfile
import os
import sys

def run_dssp_directly(pdb_path, dssp_executable="mkdssp"):
    """
    直接调用DSSP命令行提取二级结构序列
    将空格（无结构）转换为'-'
    """
    temp_out = None
    try:
        if not Path(pdb_path).exists() or Path(pdb_path).stat().st_size == 0:
            return None
            
        temp_out = tempfile.mktemp(suffix='.dssp')
        cmd = [dssp_executable, str(pdb_path), temp_out]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode != 0:
            # 尝试备用命令
            if dssp_executable == "mkdssp":
                return run_dssp_directly(pdb_path, "dssp")
            return None
        
        # 解析DSSP输出
        with open(temp_out, 'r') as f:
            content = f.read()
        
        os.unlink(temp_out)
        temp_out = None
        
        ss_sequence = []
        lines = content.split('\n')
        data_started = False
        
        for line in lines:
            if 'RESIDUE AA STRUCTURE' in line:
                data_started = True
                continue
                
            if not data_started or not line.strip() or line.startswith('!'):
                continue
            
            # 提取第17列的二级结构代码
            if len(line) > 16:
                ss_code = line[16]
                
                # 关键转换：空格 -> '-'
                if ss_code == ' ':
                    ss_code = '-'
                    
                ss_sequence.append(ss_code)
        
        return "".join(ss_sequence) if ss_sequence else None
        
    except Exception as e:
        print(f"  DSSP错误: {e}", file=sys.stderr)
        return None
    finally:
        if temp_out and os.path.exists(temp_out):
            os.unlink(temp_out)

In [2]:
pdb_file = "result.pdb"
dssp_executable = "mkdssp"   
# 调用函数
ss = run_dssp_directly(pdb_file, dssp_executable)

if ss:
    print(f"二级结构序列: {ss}")
    print(f"长度: {len(ss)}")
else:
    print("DSSP 分析失败，请检查 PDB 文件路径和 mkdssp/dssp 是否已安装")

二级结构序列: -HHHHHHHHHHHHHHHHH-SS-B-HHHHHHHHTS-HHHHHHHHHHHHHTT--EEEETTEEEETT-
长度: 65
